# Load and process training data from extractor cache


In [ ]:
import os
import datasets
import tqdm
from utils import process_extractor_data_from_cache
from minhash import dedup

def create_full_content(example):
    content = example['content']
    if isinstance(content, list):
        content = "\n".join(content)
    question = example['question']
    return {
        'full_content': f"{question}\n{content}",
    }

datapath = "/fsx/ubuntu/users/hieuman/state-aware-rag/extractor_data-v1/training_data/extractor-Claude3.7-Sonnet-20250219-v1/bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0"
files = [f for f in os.listdir(datapath) if f.endswith('.json')]
# files = files[:100]
all_data = []
for file in tqdm.tqdm(files, desc="Processing files"):
    full_path = os.path.join(datapath, file)
    file_data = process_extractor_data_from_cache(full_path)
    all_data.extend(file_data)

dataset = datasets.Dataset.from_list(all_data)
dataset = dataset.map(
    lambda x: {'content': [c.strip() for c in x['content'] if isinstance(c, str) and c.strip()]},
    num_proc=32
)
dataset = dataset.filter(
    lambda x: len(x['content']) > 0,
    num_proc=32
)
# Deduplicate the dataset
explore_dataset = dataset.filter(
    lambda x: x['type'] == 'explore',
    num_proc=32
)
reflex_dataset = dataset.filter(
    lambda x: x['type'] == 'reflect',
    num_proc=32
)
synthesize_dataset = dataset.filter(
    lambda x: x['type'] == 'synthesize',
    num_proc=32
)

explore_dataset = explore_dataset.map(
    create_full_content,
    num_proc=32
)
explore_dataset = dedup(
    column='full_content',
    data_path=None,
    num_proc=32,
    ds=explore_dataset,
    batch_size=1000,
    idx_column=None, 
    ngram=5,
    min_length=5,
    num_perm=250,
    threshold=0.7,
)
reflex_dataset = reflex_dataset.map(
    create_full_content,
    num_proc=32
)
reflex_dataset = dedup(
    column='full_content',
    data_path=None,
    num_proc=32,
    ds=reflex_dataset,
    batch_size=1000,
    idx_column=None, 
    ngram=5,
    min_length=64,
    num_perm=250,
    threshold=0.7,
)
synthesize_dataset = synthesize_dataset.map(
    create_full_content,
    num_proc=32
)
synthesize_dataset = dedup(
    column='full_content',
    data_path=None,
    num_proc=32,
    ds=synthesize_dataset,
    batch_size=1000,
    idx_column=None,
    ngram=5,
    min_length=64,
    num_perm=250,
    threshold=0.7,
)

In [ ]:
# Save the datasets
explore_dataset.save_to_disk('/fsx/ubuntu/users/hieuman/state-aware-rag/data/explore_dataset-v0')
reflex_dataset.save_to_disk('/fsx/ubuntu/users/hieuman/state-aware-rag/data/reflex_dataset-v0')
synthesize_dataset.save_to_disk('/fsx/ubuntu/users/hieuman/state-aware-rag/data/synthesize_dataset-v0')

### Convert data into converational data

In [16]:
import datasets
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from agents.prompts.extract import EXTRACT_PROMPT, ExtractOutput

def convert_to_conversational(example):
    question = example['question']
    content = example['content']
    content = "\n".join(content) if isinstance(content, list) else content
    input_content = EXTRACT_PROMPT.format(
        question=question,
        document=content,
        examples="No examples provided."
    )
    output = example['output']
    # Convert output dict to ExtractOutput
    output = ExtractOutput.model_validate(output)
    output = output.model_dump_json(indent=2)
    output_reasoning = example['reasoning']
    output_content = f"<think>{output_reasoning}</think>{output}" # Hardcoded for now, which only works for Qwen3
    messages = [
        {'role': 'user', 'content': input_content},
        {'role': 'assistant', 'content': output_content}
    ]
    return {'messages': messages}


data_files = [
    '/fsx/ubuntu/users/hieuman/state-aware-rag/data/explore_dataset-v0',
    '/fsx/ubuntu/users/hieuman/state-aware-rag/data/reflex_dataset-v0',
    '/fsx/ubuntu/users/hieuman/state-aware-rag/data/synthesize_dataset-v0'
]

data = [datasets.load_from_disk(file) for file in data_files]
dataset = datasets.concatenate_datasets(data)
dataset = dataset.map(
    convert_to_conversational,
    num_proc=32,
    remove_columns=dataset.column_names
)
# Save the final dataset
dataset.save_to_disk('/fsx/ubuntu/users/hieuman/state-aware-rag/data/data-v0')

Saving the dataset (1/1 shards): 100%|██████████| 22668/22668 [00:00<00:00, 33185.14 examples/s]


In [18]:
dataset[0]

{'messages': [{'content': 'You are a meticulous and insightful research analyst. Your primary objective is to build a comprehensive dossier of all information from the provided text that could help a user fully understand and answer their question. You prioritize thoroughness, context, and nuance. You must think step-by-step to ensure no helpful detail, however tangential, is overlooked.\n\n## Instructions: \n- Step 1: Question Deconstruction: First, carefully analyze the user\'s Question. Identify and list the primary subject, all key entities (people, organizations, concepts), and the specific information or insight the user is seeking. This is your \'search brief\'.\n- Step 2: Candidate Identification: Next, read the entire Raw Data and identify and quote ALL passages that seem potentially related to the concepts from Step 1. Be liberal and inclusive in this initial pass; we will filter and refine in the next step. If no passages appear even remotely related, state this and proceed 